# Rwanda SME Fragility: Reproducible Analysis Notebook

This notebook replaces the standalone Python scripts as the single reproducible analysis source for Paper 1. It uses CSV inputs only, including the CSV converted from the original Stata `.dta` file and the CSV files converted from the processed Excel workbooks.

Core goals:

1. Load and audit the raw Rwanda Enterprise Survey CSV.
2. Explain how raw survey contents contribute to the paper.
3. Rebuild the analysis dataset from processed CSVs.
4. Implement the mathematical fragility-index and Omega-score formulas.
5. Compare the draft formula against the supplied classification file.
6. Rerun the LASSO-logit model and regenerate tables/plots.
7. Identify new visualization and research-paper opportunities.
8. Provide optional code cells for loading external spatial and formal/informal enterprise data from official sources.

Important caution: the Word draft and processed predictor files contain some variable-label inconsistencies. This notebook makes those inconsistencies visible rather than hiding them.

In [1]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = Path("D:/Research/LASSO")

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_TABLES = ROOT / "results" / "tables"
RESULTS_FIGURES = ROOT / "results" / "figures"
RESULTS_MODELS = ROOT / "results" / "models"

for p in [RESULTS_TABLES, RESULTS_FIGURES, RESULTS_MODELS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "axes.grid": True, "grid.alpha": 0.25})
print(ROOT)

D:\Research\LASSO


## 1. Load Converted CSV Files

The raw Stata file has been converted to `data/raw/Rwanda-2023-full-data.csv`. Variable labels extracted from the Stata metadata are stored in `data/raw/Rwanda-2023-variable-labels.csv`. All processed `.xlsx` files have also been converted to `.csv`.

In [2]:
raw = pd.read_csv(DATA_RAW / "Rwanda-2023-full-data.csv")
labels = pd.read_csv(DATA_RAW / "Rwanda-2023-variable-labels.csv")
predictors = pd.read_csv(DATA_PROCESSED / "Rwanda_2023_predictors_only.csv")
classification = pd.read_csv(DATA_PROCESSED / "classification_probabilities.csv")
labeled_predictors = pd.read_csv(DATA_PROCESSED / "Labeled_Predictors_for_Rwanda_SME_Fragility_Study.csv")
omega_export = pd.read_csv(DATA_PROCESSED / "omega_score.csv")

print("Raw survey shape:", raw.shape)
print("Predictor file shape:", predictors.shape)
print("Classification file shape:", classification.shape)
print("Labeled predictors shape:", labeled_predictors.shape)
print("Omega export shape:", omega_export.shape)

Raw survey shape: (358, 355)
Predictor file shape: (358, 13)
Classification file shape: (358, 2)
Labeled predictors shape: (358, 13)
Omega export shape: (358, 15)


## 2. What the Raw `.dta`/CSV Contributes to the Paper

The converted raw survey file contributes three things that the processed files alone cannot fully provide:

- **Survey provenance and full covariate universe:** 355 variables, not just the 13 curated predictors.
- **Codebook context:** labels for survey variables such as sector, sampling region, weights, innovation, finance, e-payments, and digital presence.
- **Future research surface:** region, weights, business environment constraints, infrastructure, management, green economy, labor, performance, and finance variables can support new papers beyond the current LASSO screening article.

In [3]:
selected_codes = [
    "h1", "h2", "h5", "h8", "k82", "k162", "k3a", "k3bc", "c22b", "k33",
    "a2", "a3a", "a4a", "a6a", "b6", "wmedian", "wstrict", "wweak",
    "m1a", "j30a", "l1", "r1", "ge3"
]
label_lookup = labels.set_index("variable")["label"].to_dict()
raw_context = pd.DataFrame({
    "variable": selected_codes,
    "label": [label_lookup.get(v, "") for v in selected_codes],
    "non_missing": [raw[v].notna().sum() if v in raw.columns else np.nan for v in selected_codes],
    "missing_pct": [round(raw[v].isna().mean() * 100, 2) if v in raw.columns else np.nan for v in selected_codes],
    "unique_values": [raw[v].nunique(dropna=True) if v in raw.columns else np.nan for v in selected_codes],
})
raw_context.to_csv(RESULTS_TABLES / "table_raw_variable_context.csv", index=False)
raw_context

,variable,label,non_missing,missing_pct,unique_values
0,h1,New Products/Services Introduced Over Last 3 Yrs,358,0.00,2
1,h2,New Products/Services Also New For Thr Establi...,238,33.52,3
2,h5,"During Last 3 Yrs, Establishment Introduced Ne...",358,0.00,2
3,h8,"During Last Fiscal Year, Establishment Spent O...",358,0.00,2
4,k82,Establishment Has A Line of Credit or Loan Fro...,358,0.00,5
5,k162,"In Last FY, Did Establishment Apply For New Lo...",358,0.00,5
6,k3a,% of Working Capital Financed From Internal Fu...,358,0.00,26
7,k3bc,% of Working Capital Borrowed From Banks,358,0.00,21
8,c22b,Establishment Has Its Own Website,358,0.00,2
9,k33,Percentage of payments received using e-payments,358,0.00,34


In [4]:
missing = raw.isna().mean().sort_values(ascending=False).head(25) * 100
fig, ax = plt.subplots(figsize=(8, 6))
missing.sort_values().plot(kind="barh", ax=ax, color="#6b8e23")
ax.set_xlabel("Missing values (%)")
ax.set_title("Raw WBES variables with highest missingness")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_raw_missingness_top25.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\4214230400.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Rebuild the Analysis Dataset from CSV

This replaces the former `01_prepare_analysis_dataset.py` script. The processed predictor CSV contains the modeling variables; the classification CSV provides the current binary outcome and predicted probabilities.

In [5]:
VARIABLE_LABELS = {
    "h1": "change_permanent_employment",
    "h2": "change_temporary_employment",
    "h5": "revenue_volatility",
    "h8": "capacity_utilization_change",
    "k82": "internet_for_business",
    "k162": "external_finance_assets",
    "k3a": "delayed_client_payments",
    "k3bc": "late_supplier_payments",
    "c22b": "digital_platform_difficulty",
    "k33": "ecommerce_share",
    "a4a": "sector",
    "a6a": "firm_size",
    "b6": "manager_experience_years",
}

analysis = pd.concat([classification, predictors.rename(columns=VARIABLE_LABELS)], axis=1)
analysis["omega_score"] = analysis["fragility_prob"]
analysis["fragility_label"] = analysis["fragility_risk_flex"].map({0: "Non-fragile", 1: "Fragile"})
analysis["revenue_volatility_binary"] = (analysis["revenue_volatility"] == 2).astype(int)
analysis["digital_platform_difficulty_binary"] = (analysis["digital_platform_difficulty"] == 2).astype(int)
analysis.to_csv(DATA_PROCESSED / "analysis_dataset.csv", index=False)

analysis.head()

,fragility_risk_flex,fragility_prob,change_permanent_employment,change_temporary_employment,revenue_volatility,capacity_utilization_change,internet_for_business,external_finance_assets,delayed_client_payments,late_supplier_payments,digital_platform_difficulty,ecommerce_share,sector,firm_size,manager_experience_years,omega_score,fragility_label,revenue_volatility_binary,digital_platform_difficulty_binary
0,0,0.013295,2,NaN,2,2,1,1,70,30,1,40,2,1,5,0.013295,Non-fragile,1,0
1,0,0.013295,2,NaN,2,2,4,4,90,0,1,75,3,1,5,0.013295,Non-fragile,1,0
2,0,0.009681,1,1.0,1,2,4,4,90,0,2,2,3,1,35,0.009681,Non-fragile,0,1
3,0,0.009681,1,1.0,1,2,4,4,100,0,2,0,3,1,3,0.009681,Non-fragile,0,1
4,1,0.013295,1,-9.0,2,2,4,4,100,0,1,100,3,1,8,0.013295,Fragile,1,0


## 4. Implement the Mathematical Fragility Formula

The Word draft defines fragility as a compounding-risk condition. Let:

\[
R^F_i = \mathbb{1}(F_i < 0.25), \quad
R^D_i = \mathbb{1}(D_i < 0.30), \quad
R^E_i = \mathbb{1}(E_i < 0.10)
\]

Then:

\[
Fragility_i = \mathbb{1}\{(R^F_iR^D_i) + (R^F_iR^E_i) + (R^D_iR^E_i) \geq 1\}
\]

This means a firm is fragile if at least two of the three dimensions are high-risk. The draft also proposed Omega scores:

\[
\Omega_1 = 0.40F + 0.40D + 0.20E
\]

\[
\Omega_2 = 0.20F + 0.30D + 0.50E
\]

\[
\Omega_3 = 0.25(A + O + P + E)
\]

The exact component mapping must be codebook-verified. Below, we implement the draft version transparently and compare it to the supplied classification.

In [6]:
def yes_no_to_binary(series):
    """WBES-style recode: 1=Yes, 2=No, -9=Don't know/missing."""
    return series.map({1: 1.0, 2: 0.0, -9: np.nan})

# Draft formula components from Full_Paper_1.docx.
F = pd.concat([yes_no_to_binary(raw[c]) for c in ["h1", "h2", "h5", "h8"]], axis=1).fillna(0).mean(axis=1)
D = (yes_no_to_binary(raw["c22b"]).fillna(0) + raw["k33"].replace(-9, np.nan).fillna(0).clip(0, 100) / 100) / 2
E = raw["k3bc"].replace(-9, np.nan).fillna(0).clip(0, 100) / 100

R_F = (F < 0.25).astype(int)
R_D = (D < 0.30).astype(int)
R_E = (E < 0.10).astype(int)
fragility_formula = (((R_F * R_D) + (R_F * R_E) + (R_D * R_E)) >= 1).astype(int)

omega_1 = 0.40 * F + 0.40 * D + 0.20 * E
omega_2 = 0.20 * F + 0.30 * D + 0.50 * E
omega_3 = 0.25 * (F + yes_no_to_binary(raw["c22b"]).fillna(0) + (raw["k33"].replace(-9, np.nan).fillna(0).clip(0, 100) / 100) + E)

formula_df = pd.DataFrame({
    "financial_access_F": F,
    "digital_presence_D": D,
    "ecommerce_E": E,
    "risk_finance": R_F,
    "risk_digital": R_D,
    "risk_ecommerce": R_E,
    "fragility_formula": fragility_formula,
    "fragility_supplied": classification["fragility_risk_flex"],
    "omega_1": omega_1,
    "omega_2": omega_2,
    "omega_3": omega_3,
})
formula_df.to_csv(RESULTS_MODELS / "fragility_formula_components.csv", index=False)

agreement = (formula_df["fragility_formula"] == formula_df["fragility_supplied"]).mean()
print("Formula fragile count:", int(formula_df["fragility_formula"].sum()))
print("Supplied fragile count:", int(formula_df["fragility_supplied"].sum()))
print("Agreement:", round(agreement, 3))
pd.crosstab(formula_df["fragility_formula"], formula_df["fragility_supplied"], rownames=["Formula"], colnames=["Supplied"])

Formula fragile count: 104
Supplied fragile count: 62
Agreement: 0.765


Supplied,0,1
Formula,,
0,233,21
1,63,41


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, col, title in zip(axes, ["financial_access_F", "digital_presence_D", "ecommerce_E"], ["Financial access", "Digital presence", "E-commerce/working-capital proxy"]):
    ax.hist(formula_df.loc[formula_df.fragility_supplied == 0, col], bins=15, alpha=0.65, label="Non-fragile", color="#4c78a8")
    ax.hist(formula_df.loc[formula_df.fragility_supplied == 1, col], bins=15, alpha=0.65, label="Fragile", color="#e45756")
    ax.set_title(title)
    ax.set_xlabel("Score")
axes[0].set_ylabel("Firms")
axes[-1].legend()
fig.suptitle("Draft formula components by supplied fragility status", y=1.03)
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_formula_components_by_fragility.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\3087864689.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
ct = pd.crosstab(formula_df["fragility_formula"], formula_df["fragility_supplied"])
fig, ax = plt.subplots(figsize=(4.5, 3.8))
im = ax.imshow(ct.values, cmap="Blues")
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha="center", va="center", color="black", fontsize=12)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Supplied 0", "Supplied 1"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Formula 0", "Formula 1"])
ax.set_title("Formula vs supplied fragility classification")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_formula_vs_supplied_fragility.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\466495372.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation of Formula Comparison

The draft formula is useful theoretically, but it does **not** exactly reproduce the supplied classification file. That is important for the paper: the equations explain the intended logic, while the current empirical results depend on the supplied classification/probability export. Before submission, the team should either:

1. recover the exact original code that generated `classification_probabilities.csv`, or
2. formally adopt and report the transparent formula implemented above, then rerun all results using that reconstructed outcome.


## 4b. Recovered Exact Legacy Classification Rule

A shallow decision-tree audit recovers an exact rule that reproduces the supplied `fragility_risk_flex` classification with 100% agreement. This is the transparent computational definition of the current paper outcome.

Using official raw WBES codes:

\[
fragility\_risk\_flex_i = 1
\]

if any of the following hold:

1. \(h5_i = 1\) and \(l1_i \leq 3\)
2. \(h5_i = 2\), \(c22b_i = 1\), and \(l1_i \leq -2\)
3. \(h5_i = 2\), \(c22b_i = 2\), and either \(h1_i = 2\) or \(h2_i \neq 1\)

Otherwise, \(fragility\_risk\_flex_i = 0\).

Important construct-validity warning: the official Stata labels describe `h5` as process innovation, `c22b` as establishment website, `h1/h2` as product/service innovation, and `l1` as permanent full-time employees. This differs from some Word-draft descriptions that call `h5` revenue volatility and `c22b` digital-platform difficulty. The recovered rule is exact, but the substantive naming must be harmonized before journal submission.


In [9]:

# Recovered exact rule from the supplied classification export.
recovered_fragility = (
    ((raw["h5"] == 1) & (raw["l1"] <= 3.5)) |
    ((raw["h5"] == 2) & (raw["c22b"] == 1) & (raw["l1"] <= -2)) |
    ((raw["h5"] == 2) & (raw["c22b"] == 2) & ((raw["h1"] == 2) | (raw["h2"] != 1)))
).astype(int)

recovery_check = pd.DataFrame({
    "fragility_supplied": classification["fragility_risk_flex"],
    "fragility_recovered_rule": recovered_fragility,
    "h1": raw["h1"],
    "h2": raw["h2"],
    "h5": raw["h5"],
    "c22b": raw["c22b"],
    "l1": raw["l1"],
    "fragility_prob": classification["fragility_prob"],
})
recovery_check["match"] = recovery_check["fragility_supplied"] == recovery_check["fragility_recovered_rule"]
recovery_check.to_csv(RESULTS_MODELS / "recovered_fragility_rule_audit.csv", index=False)

recovered_summary = pd.DataFrame({
    "Measure": ["Supplied fragile firms", "Recovered-rule fragile firms", "Agreement", "Mismatches"],
    "Value": [
        int(classification["fragility_risk_flex"].sum()),
        int(recovered_fragility.sum()),
        round(recovery_check["match"].mean(), 3),
        int((~recovery_check["match"]).sum()),
    ],
})
recovered_summary.to_csv(RESULTS_TABLES / "table_recovered_classification_rule_summary.csv", index=False)
recovered_summary.to_latex(RESULTS_TABLES / "table_recovered_classification_rule_summary.tex", index=False, escape=True)

print(recovered_summary.to_string(index=False))
pd.crosstab(recovery_check["fragility_recovered_rule"], recovery_check["fragility_supplied"], rownames=["Recovered"], colnames=["Supplied"])


                     Measure  Value
      Supplied fragile firms   62.0
Recovered-rule fragile firms   62.0
                   Agreement    1.0
                  Mismatches    0.0


Supplied,0,1
Recovered,,
0,296,0
1,0,62


## 5. Descriptive Tables and Plots

This replaces the former `02_descriptive_tables.py` script and adds more figures for the research context.

In [10]:
def pct(x):
    return 100 * x.mean()

sample = pd.DataFrame({
    "Measure": ["Observations", "Fragile firms", "Non-fragile firms", "Fragility prevalence (%)", "Mean predicted probability"],
    "Value": [len(analysis), int(analysis["fragility_risk_flex"].sum()), int((1 - analysis["fragility_risk_flex"]).sum()), round(pct(analysis["fragility_risk_flex"]), 2), round(analysis["fragility_prob"].mean(), 4)],
})
sample.to_csv(RESULTS_TABLES / "table_1_sample_summary.csv", index=False)
sample.to_latex(RESULTS_TABLES / "table_1_sample_summary.tex", index=False, escape=True)

profile = (analysis.groupby("fragility_label")
    .agg(n=("fragility_risk_flex", "size"),
         revenue_volatility_pct=("revenue_volatility_binary", pct),
         digital_difficulty_pct=("digital_platform_difficulty_binary", pct),
         mean_predicted_probability=("fragility_prob", "mean"),
         mean_manager_experience=("manager_experience_years", "mean"))
    .round(3).reset_index())
profile.to_csv(RESULTS_TABLES / "table_2_fragility_profile.csv", index=False)
profile.to_latex(RESULTS_TABLES / "table_2_fragility_profile.tex", index=False, escape=True)

sample, profile

(                      Measure     Value
 0                Observations  358.0000
 1               Fragile firms   62.0000
 2           Non-fragile firms  296.0000
 3    Fragility prevalence (%)   17.3200
 4  Mean predicted probability    0.1732,
   fragility_label    n  revenue_volatility_pct  digital_difficulty_pct  \
 0         Fragile   62                  98.387                  98.387   
 1     Non-fragile  296                  29.054                  38.514   
 
    mean_predicted_probability  mean_manager_experience  
 0                       0.807                    8.823  
 1                       0.040                   27.375  )

In [11]:
fig, ax = plt.subplots(figsize=(6.5, 4))
counts = analysis["fragility_label"].value_counts().reindex(["Non-fragile", "Fragile"])
ax.bar(counts.index, counts.values, color=["#4c78a8", "#e45756"])
ax.set_ylabel("Number of firms")
ax.set_title("Fragility classification in the analysis sample")
for i, v in enumerate(counts.values):
    ax.text(i, v + 4, str(v), ha="center")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_fragility_distribution.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\3210954485.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [("Non-fragile", "#4c78a8"), ("Fragile", "#e45756")]:
    vals = analysis.loc[analysis.fragility_label == label, "fragility_prob"]
    ax.hist(vals, bins=20, alpha=0.7, label=label, color=color)
ax.set_xlabel("Predicted fragility probability")
ax.set_ylabel("Firms")
ax.set_title("Predicted probability distribution by fragility status")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_predicted_probability_distribution_rebuilt.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\3954102225.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
region_map = {1: "Kigali", 2: "Western/Northern", 3: "Southern/Eastern"}
sector_map = {1: "Manufacturing", 2: "Retail", 3: "Other services"}
size_map = {1: "Small", 2: "Medium", 3: "Large"}

context = analysis.copy()
context["region"] = raw["a3a"].map(region_map)
context["sector_label"] = raw["a4a"].map(sector_map)
context["size_label"] = raw["a6a"].map(size_map)

for col, title, fname in [
    ("region", "Fragility prevalence by region", "figure_fragility_by_region.png"),
    ("sector_label", "Fragility prevalence by sector", "figure_fragility_by_sector.png"),
    ("size_label", "Fragility prevalence by firm size", "figure_fragility_by_size.png"),
]:
    prev = context.groupby(col)["fragility_risk_flex"].mean().sort_values() * 100
    fig, ax = plt.subplots(figsize=(7, 4))
    prev.plot(kind="barh", ax=ax, color="#72b7b2")
    ax.set_xlabel("Fragile firms (%)")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(RESULTS_FIGURES / fname)
    plt.show()

sector_table = pd.crosstab(context["sector_label"], context["fragility_label"], margins=True)
sector_table.to_csv(RESULTS_TABLES / "table_3_sector_by_fragility.csv")
sector_table.to_latex(RESULTS_TABLES / "table_3_sector_by_fragility.tex", escape=True)

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\1016231135.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\1016231135.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\1016231135.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
scatter = ax.scatter(
    formula_df["financial_access_F"],
    formula_df["digital_presence_D"],
    c=analysis["fragility_risk_flex"],
    cmap="coolwarm",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.3,
)
ax.axvline(0.25, color="black", linestyle="--", linewidth=1)
ax.axhline(0.30, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Financial access score F")
ax.set_ylabel("Digital presence score D")
ax.set_title("Finance-digital risk space")
fig.colorbar(scatter, ax=ax, label="Supplied fragility")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_finance_digital_risk_space.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\3190375190.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Rerun the LASSO-Logit Model

This replaces the former `03_predictive_model.py` script. The model uses the supplied classification outcome because that is the current outcome behind Paper 1's empirical tables.

In [15]:
TARGET = "fragility_risk_flex"
FEATURES = [
    "change_permanent_employment", "change_temporary_employment", "revenue_volatility",
    "capacity_utilization_change", "internet_for_business", "external_finance_assets",
    "delayed_client_payments", "late_supplier_payments", "digital_platform_difficulty",
    "ecommerce_share", "sector", "firm_size", "manager_experience_years",
]

X = analysis[FEATURES]
y = analysis[TARGET]
numeric = X.select_dtypes(include=["number"]).columns.tolist()
categorical = [c for c in X.columns if c not in numeric]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])

model = Pipeline([
    ("prep", preprocessor),
    ("logit_lasso", LogisticRegressionCV(Cs=20, cv=5, penalty="l1", solver="liblinear", scoring="roc_auc", class_weight="balanced", max_iter=5000, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall/Sensitivity", "Specificity", "ROC-AUC", "True positives", "False positives", "True negatives", "False negatives"],
    "Value": [accuracy_score(y_test, pred), precision_score(y_test, pred, zero_division=0), recall_score(y_test, pred, zero_division=0), tn/(tn+fp), roc_auc_score(y_test, prob), tp, fp, tn, fn],
})
metrics.to_csv(RESULTS_TABLES / "table_4_lasso_test_performance.csv", index=False)
metrics.to_latex(RESULTS_TABLES / "table_4_lasso_test_performance.tex", index=False, escape=True)

feature_names = model.named_steps["prep"].get_feature_names_out()
coef = model.named_steps["logit_lasso"].coef_[0]
coef_table = pd.DataFrame({"feature": feature_names, "coefficient": coef}).assign(abs_coefficient=lambda d: d.coefficient.abs()).sort_values("abs_coefficient", ascending=False)
coef_table.to_csv(RESULTS_TABLES / "table_5_lasso_coefficients.csv", index=False)
coef_table.head(20).to_latex(RESULTS_TABLES / "table_5_lasso_coefficients.tex", index=False, escape=True)

pred_out = X_test.copy()
pred_out[TARGET] = y_test.values
pred_out["predicted_probability"] = prob
pred_out["predicted_class"] = pred
pred_out.to_csv(RESULTS_MODELS / "lasso_test_predictions.csv", index=False)

metrics, coef_table.head(12)

(               Metric      Value
 0            Accuracy   0.966667
 1           Precision   0.882353
 2  Recall/Sensitivity   0.937500
 3         Specificity   0.972973
 4             ROC-AUC   0.975929
 5      True positives  15.000000
 6     False positives   2.000000
 7      True negatives  72.000000
 8     False negatives   1.000000,
                              feature  coefficient  abs_coefficient
 8   num__digital_platform_difficulty     2.218909         2.218909
 2            num__revenue_volatility     1.933911         1.933911
 0   num__change_permanent_employment     0.529206         0.529206
 1   num__change_temporary_employment    -0.289138         0.289138
 5       num__external_finance_assets     0.116235         0.116235
 4         num__internet_for_business     0.000000         0.000000
 3   num__capacity_utilization_change     0.000000         0.000000
 6       num__delayed_client_payments     0.000000         0.000000
 7        num__late_supplier_payments     0.000

In [16]:
fpr, tpr, _ = roc_curve(y_test, prob)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, label=f"LASSO logit (AUC={roc_auc_score(y_test, prob):.3f})", color="#2f5597")
ax.plot([0, 1], [0, 1], linestyle="--", color="0.5", label="No-skill")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Held-out ROC curve")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_lasso_test_roc.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\4243026271.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
top = coef_table[coef_table["abs_coefficient"] > 0].head(10).sort_values("coefficient")
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = np.where(top["coefficient"] >= 0, "#e45756", "#4c78a8")
ax.barh(top["feature"], top["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Standardized LASSO coefficient")
ax.set_title("Selected predictors in the LASSO-logit model")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_lasso_selected_coefficients.png")
plt.show()

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\2229751712.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. What Else Should We Visualize from the Raw CSV?

The raw CSV opens up new research angles that are not yet used in Paper 1. High-value visualization candidates:

- **Spatial stratification:** fragility by `a3a` region and, if district data can be added, district-level vulnerability maps.
- **Survey weights:** compare unweighted versus weighted fragility prevalence using `wmedian`, `wstrict`, or `wweak`.
- **Finance constraints:** loan access, collateral, rejection, payment delays, and obstacle severity.
- **Digitalization stack:** website, e-payments, e-commerce, internet use, and platform difficulty.
- **Infrastructure stress:** electricity, water, transport, outages, and losses.
- **Management and productivity:** monitored performance indicators, manager experience, workforce training, and innovation.
- **Green economy and climate exposure:** green practices, energy constraints, environmental regulation, and shock adaptation.

In [18]:
# Example: weighted vs unweighted fragility prevalence by region.
weighted_context = context.copy()
weighted_context["wmedian"] = raw["wmedian"]
weighted_region = weighted_context.groupby("region").apply(
    lambda d: np.average(d["fragility_risk_flex"], weights=d["wmedian"])
).sort_values() * 100
unweighted_region = weighted_context.groupby("region")["fragility_risk_flex"].mean().reindex(weighted_region.index) * 100

comp = pd.DataFrame({"Unweighted": unweighted_region, "Weighted_wmedian": weighted_region})
fig, ax = plt.subplots(figsize=(7.5, 4.5))
comp.plot(kind="barh", ax=ax)
ax.set_xlabel("Fragile firms (%)")
ax.set_title("Weighted vs unweighted fragility prevalence by region")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "figure_weighted_unweighted_region_fragility.png")
plt.show()
comp

C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\2639674217.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_region = weighted_context.groupby("region").apply(


C:\Users\Simeon\AppData\Local\Temp\ipykernel_15812\2639674217.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Unweighted,Weighted_wmedian
region,,
Kigali,8.461538,12.947375
Western/Northern,24.324324,30.123287
Southern/Eastern,20.512821,35.376855


## 8. External Data for New Papers

Official sources worth adding:

1. **World Bank Enterprise Survey Rwanda 2023 metadata and microdata:** https://microdata.worldbank.org/index.php/catalog/6468
2. **National Institute of Statistics Rwanda Integrated Business Enterprise Survey 2023:** https://www.statistics.gov.rw/statistical-publications/business-establishment-finance-trade/business-establishment-finance-trade/integrated-business-enterprise-survey-2023
3. **NISR Establishment Census 2023:** https://www.beta.statistics.gov.rw/publication/2148
4. **NISR IBES datasource page:** https://www.statistics.gov.rw/datasource/integrated-business-enterprise-survey
5. **Rwanda administrative boundaries/geospatial data:** add official NISR or Rwanda open-data boundary files if district-level firm geography is available.

Potential new papers:

- **Paper 2: Spatial Inequality in SME Fragility.** Merge firm-level fragility with district/province boundaries, nightlights, road access, electricity/internet infrastructure, and market density.
- **Paper 3: Formal-Informal Enterprise Gaps and Fragility.** Combine WBES formal-firm data with IBES/Establishment Census formal-informal counts to show what WBES misses and how informality reshapes resilience policy.
- **Paper 4: Digital Finance, E-payments, and SME Resilience.** Focus on digital payments, websites, e-commerce, bank finance, fintech adoption, and platform barriers.
- **Paper 5: Sectoral Fragility in Manufacturing-Led Development.** Use manufacturing, infrastructure, exports, capacity utilization, and employment variables to examine industrial policy risk.

In [19]:
# Optional internet-loading scaffold. Keep False for reproducible offline execution.
RUN_EXTERNAL_DOWNLOADS = False

external_sources = {
    "wbes_rwanda_2023_catalog": "https://microdata.worldbank.org/index.php/catalog/6468",
    "nisr_ibes_2023": "https://www.statistics.gov.rw/statistical-publications/business-establishment-finance-trade/business-establishment-finance-trade/integrated-business-enterprise-survey-2023",
    "nisr_establishment_census_2023": "https://www.beta.statistics.gov.rw/publication/2148",
    "nisr_ibes_datasource": "https://www.statistics.gov.rw/datasource/integrated-business-enterprise-survey",
}

if RUN_EXTERNAL_DOWNLOADS:
    # These pages expose reports and Excel tables. Depending on the site layout,
    # pd.read_html may find tables, while direct Excel links may need manual selection.
    for name, url in external_sources.items():
        try:
            tables = pd.read_html(url)
            print(name, "tables found:", len(tables))
            if tables:
                tables[0].to_csv(DATA_RAW / f"external_{name}_table0.csv", index=False)
        except Exception as exc:
            print(name, "could not be loaded automatically:", exc)
else:
    print("External downloads are disabled. Set RUN_EXTERNAL_DOWNLOADS=True when internet access is available.")

External downloads are disabled. Set RUN_EXTERNAL_DOWNLOADS=True when internet access is available.


## 9. Bottom Line for Paper 1

Paper 1 should keep the current LASSO-logit result as a predictive model using the supplied classification export, but it should also include the mathematical index logic as the conceptual basis of fragility. The strongest next technical task is to recover or reconstruct the exact classification rule so that the paper's outcome is fully transparent and not dependent on an opaque legacy export.

# Appendix: Historical Project Progress Notes

This appendix merges the useful narrative content from the former `00_project_progress_report.ipynb`. Old code cells were intentionally omitted because they reference pre-reorganization paths such as `LASSO1/...`; the reproducible code now lives in the main body of this notebook.

# Rwanda Enterprise Survey Analysis Project - LASSO Research
## Comprehensive Progress Report and Findings

A detailed analysis of SME fragility in Rwanda using the 2023 World Bank Enterprise Survey data.

*Date: September 6, 2025*

### Project Evolution
1. Initial Analysis (Stanley):
   - LASSO regression approach [Source: `LASSO1/Stanley/logit_model_results.rtf`]
   - Basic risk classification [Source: `LASSO1/Stanley/classification.csv`]
   - Preliminary findings [Source: `LASSO1/Stanley/fragility_model_results.rtf`]

2. Enhanced Analysis (Simeon):
   - Advanced machine learning models [Source: `Rwanda-2023-full-data/omega_score_analysis/analyze_omega_extraction.py`]
   - Comprehensive data processing [Source: `Rwanda-2023-full-data/01_data_overview.py`]
   - Policy recommendations development [Source: `Rwanda-2023-full-data/omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]

### Dataset Overview
- 358 firms analyzed [Source: `Rwanda-2023-full-data.csv`]
- 3 key sectors (Manufacturing, Services, Other) [Source: `Rwanda-2023-full-data/02_visualization_patterns.py`]
- Focus on SME structural fragility [Source: `Rwanda-2023-full-data/omega_score_analysis/README_Updated.md`]
- Rich digital adoption and financial access metrics [Source: `Rwanda-2023-full-data/03_correlation_analysis.py`]

## Table of Contents

1. Project Overview and Setup
2. Initial Data Files and Models (Stanley Directory)
3. Data Exploration and Processing
4. Omega Score Analysis Pipeline
5. Enhanced Analysis Framework
6. Key Findings and Visualizations

*Note: This presentation walks through the progression from initial data to final analysis*

## Detailed Methodology

### 1. Data Processing Pipeline
- Comprehensive cleaning procedures [Source: `01_data_overview.py`]
- Missing value treatment (97.8% → 99.8% completeness) [Source: `omega_score_analysis/detailed_omega_analysis.py`]
- Variable standardization [Source: `omega_score_analysis/analyze_omega_extraction.py`]
- Quality validation checks [Source: `omega_score_analysis/validation_analysis.py`]

### 2. Enhanced Omega Score Calculation
Three distinct methods implemented [Source: `omega_score_analysis/omega_score_investigation_fixed.py`]:

1. **Weighted Financial Technology Score**
   ```
   Score = 0.4 * Financial_Access + 0.4 * Digital_Presence + 0.2 * E-commerce
   ```

2. **E-Commerce Focused Score**
   ```
   Score = 0.5 * E-commerce + 0.3 * Digital_Presence + 0.2 * Financial_Access
   ```

3. **Balanced Technology Score**
   ```
   Score = 0.25 * (Financial_Tools + Online_Presence + E-payments + E-commerce)
   ```

### 3. Fragility Risk Classification
Risk assessment based on three dimensions [Source: `omega_score_analysis/README_Omega_Data_Extraction.md`]:

1. **Financial Access Score (40% weight)**
   - Variables: h1, h2, h5, h8 (checking account, overdraft, loans)
   - High Risk Threshold: score < 0.25

2. **Digital Presence Score (40% weight)**
   - Website presence (c22b)
   - E-payment usage (k33)
   - High Risk Threshold: score < 0.30

3. **E-Commerce Score (20% weight)**
   - Variable: k3bc (% sales through e-commerce)
   - High Risk Threshold: score < 0.10

### 4. Machine Learning Models
Multiple models implemented and compared [Source: `omega_score_analysis/omega_score_investigation.py`]:

1. **Gradient Boosting Machine (GBM)**
   - Best performance: AUC = 0.85
   - Accuracy = 0.82

2. **Random Forest**
   - Strong feature insights
   - AUC = 0.84

3. **Neural Network**
   - Complex pattern detection
   - AUC = 0.83

### 5. Validation Framework
- 5-fold cross-validation [Source: `omega_score_analysis/validation_analysis.py`]
- Feature importance stability analysis [Source: `omega_score_analysis/detailed_omega_analysis.py`]
- Robustness checks [Source: `omega_score_analysis/analyze_omega_extraction.py`]
- Sensitivity analysis [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]

## Manufacturing Sector Analysis

### Key Statistics [Source: `02_visualization_patterns.py`]
- 33.5% of sample (120 firms)
- 14.2% export directly
- Largest employment share across sectors

### Distinctive Characteristics [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]
1. **Infrastructure Dependencies**
   - Higher electricity access challenges
   - Greater need for reliable services
   - Complex infrastructure requirements

2. **Workforce Development**
   - Increased need for skilled workers
   - Higher training requirements
   - Specialized skill demands

3. **Business Operations**
   - Higher fixed asset investments
   - More quality certifications
   - Complex regulatory compliance

### Performance Metrics [Source: `03_correlation_analysis.py`]
1. **Export Activity**
   - 14.2% direct export rate
   - Higher than other sectors
   - International market presence

2. **Technology Adoption**
   - Above-average digital presence
   - Strong e-payment adoption
   - Infrastructure-dependent operations

3. **Financial Access**
   - Higher credit requirements
   - Significant working capital needs
   - Asset-based financing patterns

### Policy Implications [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]
1. **Infrastructure Support**
   - Power reliability improvements
   - Service quality enhancement
   - Infrastructure development

2. **Skills Development**
   - Training program support
   - Workforce development
   - Technical skill enhancement

3. **Export Promotion**
   - Market access support
   - Quality certification assistance
   - Trade facilitation

## 1. Project Overview and Setup

### Project Structure
```
LASSO1/
├── Stanley/           # Initial project files and requirements
├── Simeon/           # Implementation and analysis
│   └── Rwanda-2023-full-data/
│       └── omega_score_analysis/
└── presentation.ipynb # This file
```

### Initial Project Requirements
- Analyze Rwanda 2023 Enterprise Survey data
- Focus on SME structural fragility 
- Identify risk assessment factors
- Develop predictive models
- Create comprehensive documentation

## 2. Initial Data Files and Models (Stanley Directory)

### Key Files Overview:
1. **Classification Files**
   - classification.csv
   - classification 1.csv
   - classification 11.csv
   
2. **Model Results**
   - logit_model_results.rtf
   - fragility_model_results.rtf
   - marginal_effects.rtf
   
3. **Visualizations**
   - Model fit ROC curves
   - Graph 1 (methodology visualization)
   
4. **Data Files**
   - Labeled_Predictors_for_Rwanda_SME_Fragility_Study.csv
   - Rwanda-2023-full-data.dta

### Initial Model Results

The Stanley directory contains key model outputs:

1. **Fragility Model:**
   - Logistic regression approach
   - ROC curve analysis for model fit
   - Variable importance assessment
   
2. **Classification Results:**
   - Multiple iterations of classification
   - Refinements based on model performance
   - Predicted probabilities by class

3. **Marginal Effects:**
   - Impact analysis of key variables
   - Statistical significance assessment

## 3. Data Exploration and Processing

The data exploration phase involved several key scripts:

1. **01_data_overview.py**
   - Basic statistics
   - Sample composition
   - Key indicators analysis
   
2. **02_visualization_patterns.py**
   - Pattern visualization
   - Distribution analysis
   - Sector/size breakdowns
   
3. **03_correlation_analysis.py**
   - Variable relationships
   - Key factor identification
   
4. **04_export_curated_data.py**
   - Data curation
   - Documentation

## 4. Omega Score Analysis Pipeline

The omega score analysis involved multiple stages:

1. **Initial Extraction**
   - Data subset identification
   - Variable selection
   - Initial scoring implementation
   
2. **Data Cleaning**
   - Missing value resolution
   - Standardization
   - Outlier handling
   
3. **Enhanced Analysis**
   - Multiple imputation strategies
   - Validation checks
   - Quality metrics

## Detailed Analysis of Key Findings

### 1. Enterprise Structure Analysis
Based on our visualizations and data analysis:

- **Firm Size Distribution**
  * Small firms (5-19 employees): 47.8%
  * Medium firms (20-99 employees): 36.3%
  * Large firms (100+ employees): 15.9%
  * Implications: Strong representation of SMEs in the sample

- **Sector Distribution**
  * Manufacturing: 33.5%
  * Services: 27.9%
  * Other sectors: 38.5%
  * Key insight: Balanced sector representation

### 2. Risk Distribution Analysis
From the Omega Score results:

- **Risk Classification**
  * Low Risk (0): 82.7% (296 firms)
  * High Risk (1): 17.3% (62 firms)
  * Notable: Strong overall resilience in the sample

- **Risk Factors**
  * Financial Access
  * Digital Presence
  * Market Integration
  * Operational Efficiency

### 3. Business Environment Challenges
Top obstacles identified:

1. **Access to Finance (96 firms)**
   * Most significant challenge
   * Affects both growth and stability
   * Critical for risk assessment

2. **Infrastructure**
   * Electricity: 49 firms
   * Transport: 15 firms
   * Impact on operational efficiency

3. **Regulatory Environment**
   * Customs/trade: 42 firms
   * Business licensing: 13 firms
   * Labor regulations: 28 firms

### 4. Technology Adoption Patterns
Analysis of digital presence and technology usage:

- **Digital Infrastructure**
  * Website presence
  * E-payment systems
  * Digital communication tools
  * E-commerce adoption

- **Sector-Specific Patterns**
  * Manufacturing: Higher infrastructure dependency
  * Services: Greater digital presence
  * Other sectors: Mixed adoption patterns

### 5. Policy Implications
Based on the comprehensive analysis:

1. **Financial Inclusion**
   * Targeted SME financing programs
   * Alternative lending mechanisms
   * Risk-sharing facilities

2. **Digital Transformation**
   * Technology adoption support
   * Digital skills development
   * E-commerce enablement

3. **Infrastructure Development**
   * Power supply reliability
   * Transport network efficiency
   * Digital infrastructure

### Visualization Insights

#### 1. Risk Distribution Analysis
The bar chart clearly shows the distribution of firm risk levels:
- The majority of firms (296) are classified as low-risk (Level 0)
- A significant minority (62 firms) are classified as high-risk (Level 1)
- This 82.7% to 17.3% split suggests overall resilience in the sample

#### 2. Key Risk Metrics Distribution
The boxplot analysis reveals:
- Financial access scores show moderate variation
- Digital transformation metrics are relatively consistent
- Some outliers in specific metrics indicate opportunity for targeted interventions

#### Implications for Policy Making

1. **Targeted Support**
   - Focus on the 17.3% high-risk firms
   - Develop specific interventions for outlier cases
   - Create graduated support programs based on risk levels

2. **Success Factors**
   - Study characteristics of low-risk firms
   - Identify replicable practices
   - Develop best practice guidelines

3. **Monitoring Framework**
   - Regular risk assessment updates
   - Track intervention effectiveness
   - Adjust support programs based on outcomes

### Omega Score Quality Metrics

The analysis achieved significant improvements:

1. **Data Completeness**
   - Initial: 97.8%
   - Final: 99.8%
   
2. **Missing Values**
   - Reduced from 120 to 12
   - Group-based imputation strategy
   
3. **Derived Variables**
   - Financial access score
   - Digitalization score
   - Firm maturity metrics

## 5. Enhanced Analysis Framework

The enhanced analysis framework includes:

1. **Data Processing Pipeline**
   - Standardized cleaning procedures
   - Quality validation checks
   - Documentation generation
   
2. **Modeling Framework**
   - Multiple model comparisons
   - Cross-validation
   - Performance metrics
   
3. **Documentation Structure**
   - Methodology documentation
   - Results summary
   - Validation reports

## 6. Key Findings and Visualizations

### Sample Composition
- 358 firms analyzed
- 3 key sectors represented
- Size distribution from small to large enterprises

### Performance Metrics
- Financial access scores
- Digitalization adoption
- Infrastructure reliability
- Business obstacle assessment

### Risk Assessment
- Fragility indicators identified
- Risk factors quantified
- Sector-specific patterns

### Next Steps

1. **Publication Preparation**
   - Compile methodology documentation
   - Prepare results visualization
   - Draft research paper
   
2. **Model Refinement**
   - Incorporate feedback
   - Enhanced validation
   - Additional robustness checks
   
3. **Policy Recommendations**
   - SME support strategies
   - Risk mitigation approaches
   - Implementation guidelines

## Thank You

This presentation summarizes the progression from initial data and requirements through comprehensive analysis and findings. We're ready to move forward with publication preparation and policy recommendations.

Key achievements:
- Comprehensive data analysis pipeline
- Enhanced omega score methodology 
- Robust validation framework
- Clear policy implications

Questions and discussion welcome!

## Conclusions and Next Steps

### Key Achievements

1. **Methodology Enhancement** [Source: `omega_score_analysis/README_Updated.md`]
   - Improved omega score calculation
   - Advanced machine learning implementation
   - Robust validation framework
   - Comprehensive documentation

2. **Sector-Specific Insights** [Source: `02_visualization_patterns.py`]
   - Manufacturing sector deep dive
   - Technology adoption patterns
   - Infrastructure dependencies
   - Workforce requirements

3. **Risk Assessment Framework** [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]
   - Multi-dimensional approach
   - Validated classification method
   - Actionable indicators
   - Policy-relevant metrics

### Publication Preparation [Source: `omega_score_analysis/README_Omega_Data_Extraction.md`]

1. **Research Paper Structure**
   - Methodology documentation
   - Results visualization
   - Policy implications
   - Sector-specific recommendations

2. **Technical Documentation**
   - Model specifications
   - Validation procedures
   - Code documentation
   - Replication guidelines

3. **Policy Framework**
   - Implementation roadmap
   - Resource requirements
   - Timeline planning
   - Impact assessment

### Future Research Directions [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]

1. **Longitudinal Studies**
   - Technology adoption trends
   - Policy effectiveness tracking
   - Sector evolution patterns

2. **Comparative Analysis**
   - Regional comparisons
   - Sector benchmarking
   - Best practice identification

3. **Impact Assessment**
   - Policy intervention effects
   - Support program outcomes
   - Technology adoption impact

This comprehensive analysis has provided robust evidence for policy development while establishing a framework for ongoing research in SME development and risk assessment.

# Deep Dive Analysis Sections

## 1. Digital Transformation Patterns

### A. E-Commerce Adoption Analysis
- **Current State**
  * Percentage of firms with online sales
  * Revenue share from digital channels
  * Platform utilization patterns

- **Barriers to Adoption**
  * Technical infrastructure
  * Digital skills gap
  * Market readiness

- **Success Factors**
  * Digital infrastructure quality
  * Staff training programs
  * Integration with traditional channels

### B. Digital Payment Systems
- **Current Usage**
  * Mobile money adoption rates
  * Digital banking penetration
  * Cross-border payment solutions

- **Impact Analysis**
  * Transaction cost reduction
  * Market access improvement
  * Working capital efficiency

## 2. Financial Access Deep Dive

### A. Credit Access Patterns
- **Formal Banking Relationships**
  * Account ownership
  * Credit line access
  * Collateral requirements

- **Alternative Finance**
  * Microfinance utilization
  * Mobile lending platforms
  * Peer-to-peer lending

### B. Working Capital Management
- **Current Practices**
  * Inventory management
  * Receivables financing
  * Supplier credit utilization

- **Innovation Opportunities**
  * Supply chain financing
  * Invoice factoring
  * Digital lending platforms

## 3. Sector-Specific Technology Integration

### A. Manufacturing Sector
- **Automation Level**
  * Current automation state
  * Investment plans
  * Skills requirements

- **Digital Tools**
  * Inventory management systems
  * Production planning software
  * Quality control systems

### B. Service Sector
- **Customer Interface**
  * Digital booking systems
  * CRM implementation
  * Online service delivery

- **Operational Efficiency**
  * Resource scheduling
  * Service tracking
  * Performance monitoring

## 4. Risk Mitigation Strategies

### A. Financial Risk Management
- **Credit Risk**
  * Assessment methods
  * Monitoring systems
  * Mitigation strategies

- **Market Risk**
  * Exposure analysis
  * Hedging practices
  * Diversification strategies

### B. Operational Risk
- **Process Risk**
  * Quality control systems
  * Standard operating procedures
  * Employee training programs

- **Technology Risk**
  * System redundancy
  * Cyber security measures
  * Data protection protocols

## 5. Policy Framework Development

### A. Support Program Design
- **Financial Support**
  * Credit guarantee schemes
  * Grant programs
  * Tax incentives

- **Technical Assistance**
  * Training programs
  * Advisory services
  * Technology adoption support

### B. Implementation Strategy
- **Phased Approach**
  * Short-term interventions
  * Medium-term programs
  * Long-term initiatives

- **Monitoring Framework**
  * Key performance indicators
  * Impact assessment methods
  * Feedback mechanisms

### Digital Transformation Analysis Insights

#### 1. Digital Adoption Distribution [Source: `omega_score_analysis/analyze_omega_extraction.py`]
- **Mean Score**: 1.11 out of 2.00
- **Distribution Pattern**:
  * 25th percentile: 1.00
  * Median: 1.00
  * 75th percentile: 2.00
  * Suggests moderate digital adoption with room for improvement

#### 2. Digital Maturity Levels [Source: `omega_score_analysis/detailed_omega_analysis.py`]
- **Basic Adoption (Score 0-1)**: Primary digital tools and basic connectivity
- **Advanced Adoption (Score 1-2)**: Integrated digital systems and e-commerce capabilities

#### 3. Strategic Implications [Source: `omega_score_analysis/FINAL_ANALYSIS_SUMMARY.md`]

1. **Digital Divide**
   - Clear separation between basic and advanced adopters
   - Opportunity for targeted intervention programs
   - Need for staged digital transformation support

2. **Investment Priorities**
   - Focus on moving firms from basic to advanced adoption
   - Infrastructure support for lagging firms
   - Skill development programs aligned with adoption levels

3. **Policy Recommendations**
   - Tiered support system based on current digital maturity
   - Incentive structure for digital transformation
   - Technical assistance programs for implementation